# Brain-to-Text: Skip-Diphone + Temporal Smoothness

**Project**: Context-Aware Neural Speech Decoding with Skip-Diphone Auxiliary Supervision and Temporal Smoothness Regularization  
**Author**: Jiayu (Jarrod) Yang — Introduction to Deep Learning, Spring 2026  
**Proposal**: [../docs/proposal.pdf](../docs/proposal.pdf)

---

## Background

This project builds on the **DCoND** framework (Li et al., arXiv:2411.10657), which improves
neural speech decoding by predicting *diphones* (z_{t-1}→z_t) instead of isolated phonemes,
then marginalizing back to recover per-frame phoneme probabilities.

Two extensions are tested:

| Extension | Hypothesis |
|-----------|------------|
| **Skip-diphone auxiliary head** (z_{t-2}→z_t) | Speech motor planning encodes context beyond the immediately preceding phoneme |
| **Temporal smoothness loss** on marginalized phoneme probs | Reduces frame-level jitter and spurious CTC insertions in noisy intracortical recordings |

## Training Objective (Eq. 1)

$$\mathcal{L}_{\text{total}} = \mathcal{L}^{\text{CTC}}_{\text{phoneme}} + \alpha\,\mathcal{L}^{\text{CTC}}_{\text{std-diphone}} + \beta\,\mathcal{L}^{\text{CTC}}_{\text{skip-diphone}} + \lambda\,\mathcal{L}_{\text{smooth}}$$

$$\mathcal{L}_{\text{smooth}} = \frac{1}{B}\sum_i \frac{1}{T_i-1}\sum_{t=2}^{T_i}\|p_t - p_{t-1}\|_2^2$$

## Ablation Variants

| Variant | Components |
|---------|------------|
| A | Mono CTC only (NPTL baseline) |
| B | Diphone + marginalization (DCoND) |
| C | B + smoothness loss |
| D | B + skip-diphone auxiliary head |
| E | B + skip-diphone + smoothness (full model) |

---

This notebook reports:
1. Ablation table (variants A–E): PER and WER after 3-gram LM decoding
2. Smoothness weight λ sweep (variants C and E)
3. Skip-diphone weight β sweep (variant D)
4. WER acoustic_scale × blank_penalty sweep (heatmap)
5. Validation loss training curves per variant

All cells discover runs from `../experiments/variant_*` automatically. If a
run is missing, its row is filled with `NaN` and the plot is skipped.


In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXP_ROOT = Path('../experiments')
RUN_RE = re.compile(r'variant_(?P<variant>[A-E])_alpha(?P<alpha>[0-9.]+)_beta(?P<beta>[0-9.]+)_lam(?P<lam>[0-9.eE+-]+)')

def list_runs():
    if not EXP_ROOT.exists():
        return []
    runs = []
    for d in sorted(EXP_ROOT.iterdir()):
        if not d.is_dir():
            continue
        m = RUN_RE.fullmatch(d.name)
        if not m:
            continue
        loss_path = d / 'loss.json'
        if not loss_path.exists():
            continue
        log = json.loads(loss_path.read_text())
        if not log:
            continue
        best_per = min(e['val_per'] for e in log)
        runs.append({
            'run': d.name,
            'variant': m.group('variant'),
            'alpha': float(m.group('alpha')),
            'beta': float(m.group('beta')),
            'lambda': float(m.group('lam')),
            'best_per': best_per,
            'log': log,
        })
    return runs

runs = list_runs()
print(f'Found {len(runs)} runs')
for r in runs:
    print(f"  {r['run']:<60s} best PER {r['best_per']*100:6.2f}%")

## 1. Ablation Table (A–E)

Picks the run with the lowest best-PER for each variant. WER is read from
`<run>/wer_summary.json` if it exists (written by `decode.py` runs you
perform manually), or the minimum of `<run>/wer_sweep.csv` written by
`scripts/wer_sweep.py`.

In [ ]:
def best_run_per_variant(runs):
    by_var = {}
    for r in runs:
        v = r['variant']
        if v not in by_var or r['best_per'] < by_var[v]['best_per']:
            by_var[v] = r
    return by_var

def read_wer(run_dir):
    p = EXP_ROOT / run_dir
    summary = p / 'wer_summary.json'
    if summary.exists():
        return json.loads(summary.read_text()).get('wer', np.nan)
    csv_path = p / 'wer_sweep.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if not df.empty:
            return float(df['WER'].min())
    return np.nan

DESCRIPTIONS = {
    'A': 'Mono CTC (NPTL baseline)',
    'B': 'Diphone + marginalization (DCoND)',
    'C': 'B + smoothness',
    'D': 'B + skip-diphone aux',
    'E': 'B + skip-diphone + smoothness (full)',
}

best = best_run_per_variant(runs)
rows = []
for v in 'ABCDE':
    r = best.get(v)
    if r is None:
        rows.append({'Variant': v, 'Description': DESCRIPTIONS[v],
                     'Run': '—', 'PER %': np.nan, 'WER %': np.nan,
                     'lambda': np.nan, 'alpha': np.nan, 'beta': np.nan})
        continue
    rows.append({
        'Variant': v,
        'Description': DESCRIPTIONS[v],
        'Run': r['run'],
        'PER %': r['best_per'] * 100,
        'WER %': read_wer(r['run']) * 100,
        'lambda': r['lambda'], 'alpha': r['alpha'], 'beta': r['beta'],
    })

df = pd.DataFrame(rows)
df

## 2. λ Sweep (Smoothness Weight)

Reads all `variant_C_*` and `variant_E_*` runs and plots best PER vs. λ.

In [ ]:
def lambda_curve(runs, variant):
    pairs = [(r['lambda'], r['best_per'] * 100) for r in runs if r['variant'] == variant]
    pairs.sort()
    return pairs

fig, ax = plt.subplots(figsize=(6, 4))
for variant, marker in [('C', 'o'), ('E', 's')]:
    pairs = lambda_curve(runs, variant)
    if not pairs:
        continue
    xs, ys = zip(*pairs)
    ax.plot(xs, ys, marker=marker, label=f'Variant {variant}')
ax.set_xscale('symlog', linthresh=1e-4)
ax.set_xlabel('lambda (smoothness weight)')
ax.set_ylabel('Best PER (%)')
ax.set_title('Effect of smoothness weight on PER')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('../experiments/lambda_sweep.pdf', bbox_inches='tight')
plt.show()

## 3. β Sweep (Skip-Diphone Weight)

Variant D with λ=0 and β ∈ {0.05, 0.1, 0.2, 0.3} (run via
`scripts/beta_sweep.sh`). Confirms whether the skip-diphone head
contributes at the right weight rather than just regularizing.

In [ ]:
d_runs = [r for r in runs if r['variant'] == 'D']
d_runs.sort(key=lambda r: r['beta'])

if d_runs:
    xs = [r['beta'] for r in d_runs]
    ys = [r['best_per'] * 100 for r in d_runs]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xs, ys, marker='D', color='C2')
    ax.set_xlabel('beta (skip-diphone CTC weight)')
    ax.set_ylabel('Best PER (%)')
    ax.set_title('Variant D: PER vs. beta')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../experiments/beta_sweep.pdf', bbox_inches='tight')
    plt.show()
    pd.DataFrame({'beta': xs, 'PER %': ys})
else:
    print('No Variant D runs found yet. Run scripts/beta_sweep.sh first.')

## 4. WER Sweep Heatmap (acoustic_scale × blank_penalty)

Reads `<run>/wer_sweep.csv` written by `scripts/wer_sweep.py` and plots
WER as a heatmap. WFST acoustic scale must usually be retuned per
acoustic model, otherwise PER gains do not show up in WER.

In [ ]:
VARIANT_FOR_HEATMAP = 'E'

best_E = best.get(VARIANT_FOR_HEATMAP)
if best_E is not None:
    csv_path = EXP_ROOT / best_E['run'] / 'wer_sweep.csv'
    if csv_path.exists():
        sweep = pd.read_csv(csv_path)
        pivot = sweep.pivot(index='blank_penalty', columns='acoustic_scale', values='WER') * 100
        fig, ax = plt.subplots(figsize=(7, 4))
        im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='viridis_r')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f'{r:.2f}' for r in pivot.index])
        ax.set_xlabel('acoustic_scale')
        ax.set_ylabel('blank_penalty')
        ax.set_title(f'WER % heatmap (variant {VARIANT_FOR_HEATMAP}: {best_E["run"]})')
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, f'{pivot.values[i, j]:.1f}',
                        ha='center', va='center', color='white', fontsize=9)
        plt.colorbar(im, ax=ax, label='WER %')
        plt.tight_layout()
        plt.savefig('../experiments/wer_heatmap.pdf', bbox_inches='tight')
        plt.show()
    else:
        print(f'No wer_sweep.csv at {csv_path}. Run scripts/wer_sweep.py first.')
else:
    print(f'No variant {VARIANT_FOR_HEATMAP} runs found yet.')

## 5. Training Curves

Validation CTC loss (left) and PER (right) per variant, using the best run for each variant.

In [ ]:
fig, (ax_l, ax_p) = plt.subplots(1, 2, figsize=(11, 4))
for v in 'ABCDE':
    r = best.get(v)
    if r is None:
        continue
    log = r['log']
    epochs = [e['epoch'] for e in log]
    val_loss = [e['val_loss'] for e in log]
    val_per = [e['val_per'] * 100 for e in log]
    ax_l.plot(epochs, val_loss, label=f'Variant {v}')
    ax_p.plot(epochs, val_per, label=f'Variant {v}')

ax_l.set_xlabel('Epoch'); ax_l.set_ylabel('Validation CTC Loss')
ax_l.grid(True, alpha=0.3); ax_l.legend()
ax_p.set_xlabel('Epoch'); ax_p.set_ylabel('Validation PER (%)')
ax_p.grid(True, alpha=0.3); ax_p.legend()
plt.tight_layout()
plt.savefig('../experiments/training_curves.pdf', bbox_inches='tight')
plt.show()